# House Price Prediction with Linear Regression

**Oasis Infobyte Data Analytics Internship — Level 2, Task 1**

This notebook predicts house prices using Linear Regression on the Ames Housing dataset and evaluates the model with regression metrics and diagnostic plots.


## 1. Import Libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 2. Load the Ames Housing Dataset


In [ ]:
housing = fetch_openml(
    name="house_prices",
    as_frame=True,
    parser="auto"
)

df = housing.frame.copy()

print("Dataset shape:", df.shape)
display(df.head())


## 3. Initial EDA — Structure and Data Types


In [ ]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
missing_summary = df.isna().sum().sort_values(ascending=False)
display(missing_summary.head(20).to_frame("missing_values"))

print("\nDescriptive statistics:")
display(df.describe(include="all").T.head(30))


## 4. Target Variable — SalePrice


In [ ]:
df["SalePrice"] = pd.to_numeric(df["SalePrice"], errors="coerce")

print(df["SalePrice"].describe())

plt.figure(figsize=(9, 5))
sns.histplot(df["SalePrice"], kde=True)
plt.title("Distribution of House Sale Prices")
plt.xlabel("Sale Price")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## Observation

SalePrice is the target variable. Its distribution is inspected before modelling because house-price data can be skewed and may contain high-value observations.

The target is excluded from the feature matrix so that the model only learns from property attributes.


## 5. Feature Selection Discussion

The Ames dataset contains many property characteristics. Features such as:

- **OverallQual** — overall material and finish quality
- **GrLivArea** — above-ground living area
- **GarageCars** — garage capacity
- **YearBuilt** — property age
- **TotalBsmtSF** — basement area
- **Neighborhood** — location-related information

are plausible predictors because they describe property quality, size, age, facilities and location.

For a reproducible baseline, the notebook uses a focused set of numeric and categorical predictors rather than automatically including every available column. This also makes coefficient interpretation easier.


In [ ]:
selected_features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "YearBuilt",
    "TotalBsmtSF",
    "FullBath",
    "BedroomAbvGr",
    "TotRmsAbvGrd",
    "GarageArea",
    "Neighborhood",
    "HouseStyle",
    "CentralAir"
]

target = "SalePrice"

model_df = df[selected_features + [target]].copy()

print("Selected features:")
display(pd.DataFrame({"Feature": selected_features}))


## 6. Correlation Heatmap


In [ ]:
numeric_for_corr = model_df.select_dtypes(include=np.number)

corr = numeric_for_corr.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Correlation Heatmap — Numeric Features")
plt.tight_layout()
plt.show()

print("Correlations with SalePrice:")
display(
    corr["SalePrice"]
    .sort_values(ascending=False)
    .to_frame("correlation")
)


## 7. Train/Test Split — 80/20


In [ ]:
X = model_df.drop(columns=target)
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 8. Preprocessing

The preprocessing pipeline separates numeric and categorical variables.

**Numeric variables**
- Median imputation
- Standardisation

**Categorical variables**
- Most-frequent imputation
- One-Hot Encoding

Using a pipeline ensures that preprocessing parameters are learned only from the training data.


In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 9. Linear Regression Model


In [ ]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_test)

mse = mean_squared_error(y_test, linear_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, linear_pred)

linear_metrics = pd.DataFrame({
    "Metric": ["MSE", "RMSE", "R²"],
    "Value": [mse, rmse, r2]
})

display(linear_metrics)


## 10. Actual vs Predicted Prices


In [ ]:
plt.figure(figsize=(8, 7))
sns.scatterplot(x=y_test, y=linear_pred, alpha=0.65)

minimum = min(y_test.min(), linear_pred.min())
maximum = max(y_test.max(), linear_pred.max())

plt.plot([minimum, maximum], [minimum, maximum], linestyle="--")
plt.title("Actual vs Predicted House Prices")
plt.xlabel("Actual Sale Price")
plt.ylabel("Predicted Sale Price")
plt.tight_layout()
plt.show()


## Interpretation

Points closer to the diagonal reference line indicate predictions that are closer to the actual sale prices. Large deviations from the line represent larger prediction errors.


## 11. Residual Plot


In [ ]:
residuals = y_test - linear_pred

plt.figure(figsize=(10, 5))
sns.scatterplot(x=linear_pred, y=residuals, alpha=0.65)
plt.axhline(0, linestyle="--")
plt.title("Residual Plot")
plt.xlabel("Predicted Sale Price")
plt.ylabel("Residual (Actual - Predicted)")
plt.tight_layout()
plt.show()


## Residual Interpretation

A useful baseline regression model generally has residuals distributed around zero without a strong systematic pattern.

A visible curve, funnel shape or other structure can indicate non-linearity, unequal variance or other model limitations.


## 12. Coefficient Analysis


In [ ]:
fitted_preprocessor = linear_model.named_steps["preprocessor"]
fitted_regression = linear_model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()
coefficients = fitted_regression.coef_

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

coef_df["AbsoluteCoefficient"] = coef_df["Coefficient"].abs()

positive_coefficients = (
    coef_df[coef_df["Coefficient"] > 0]
    .sort_values("Coefficient", ascending=False)
    .head(10)
)

negative_coefficients = (
    coef_df[coef_df["Coefficient"] < 0]
    .sort_values("Coefficient")
    .head(10)
)

print("Highest positive coefficients:")
display(positive_coefficients[["Feature", "Coefficient"]])

print("Highest negative coefficients:")
display(negative_coefficients[["Feature", "Coefficient"]])


## Coefficient Interpretation

A positive coefficient indicates that, holding the other encoded features constant within the linear model, an increase in that feature is associated with a higher predicted price.

A negative coefficient indicates an association with a lower predicted price.

For categorical variables, each one-hot coefficient is interpreted relative to the omitted reference category. Because numeric features are standardized in this pipeline, coefficient magnitude can also be compared more directly across the numeric predictors.


## 13. Bonus — Ridge Regression


In [ ]:
ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=10.0))
])

ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)

ridge_mse = mean_squared_error(y_test, ridge_pred)
ridge_rmse = np.sqrt(ridge_mse)
ridge_r2 = r2_score(y_test, ridge_pred)

comparison = pd.DataFrame([
    {
        "Model": "Linear Regression",
        "MSE": mse,
        "RMSE": rmse,
        "R²": r2
    },
    {
        "Model": "Ridge Regression",
        "MSE": ridge_mse,
        "RMSE": ridge_rmse,
        "R²": ridge_r2
    }
])

display(comparison.round(4))


## 14. Final Conclusion

This project demonstrates an end-to-end house-price regression workflow.

The analysis begins with data inspection and feature selection, then applies missing-value handling and One-Hot Encoding before fitting a Linear Regression model.

MSE, RMSE and R² provide complementary views of predictive performance, while the actual-vs-predicted and residual plots provide model diagnostics.

Coefficient analysis provides an interpretable view of which encoded features have the strongest positive and negative relationships with predicted price.

The optional Ridge comparison demonstrates how regularisation can be used when a linear model contains many correlated or one-hot encoded predictors.


## 15. Save Model Comparison


In [ ]:
comparison.to_csv("house_price_model_comparison.csv", index=False)

coef_df.to_csv("linear_regression_coefficients.csv", index=False)

print("Saved:")
print("- house_price_model_comparison.csv")
print("- linear_regression_coefficients.csv")
